<a href="https://colab.research.google.com/github/Brandeis-Visual-Analytics/COSI-165b-labs/blob/main/lab7/lab_7_cnn_resnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 7: CNNs, Residual Connections, and Batch Normalization

**REMEMBER TO File > Save a Copy in Drive IF YOU ARE USING COLAB**

Adapted from 10_5_Convolution_For_MNIST, 11_2_Residual_Networks, and 11_3_Batch_Normalization of UDL Book labs.

In this lab you'll build convolutional networks to classify MNIST digits, and see how residual connections and batch normalization change the way a deeper network trains:

* **Part 1: CNNs.** You'll build a CNN with 2 convolutional layers and one with 6, train both, and compare their training curves.
* **Part 2: Residual connections.** You'll add residual connections to the 6-layer network (chapter 11 of the book) and compare again.
* **Part 3: Batch normalization.** You'll add batch normalization to the residual network and compare one last time.

Every network uses layers that are built into PyTorch, and every network is trained in exactly the same way, so the only thing that changes between them is the architecture.

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and write code to complete the classes. There are also questions interspersed in the text.

If you are using Google Colab, you can change your runtime to an instance with GPU support to speed up training (Runtime > Change runtime type > T4 GPU). If you do this, the cell below should output ``device(type='cuda')``. Each network should train in under a minute on a GPU, and in a minute or two without one.

In [ ]:
import time
import numpy as np
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Load the data

MNIST has 60,000 training images and 10,000 test images of handwritten digits. Each one is a 28x28 grayscale image. To keep training to a minute or two per network, we'll train on a random subset of 10,000 of the training images, and we'll load the images into memory all at once instead of using a data loader.

In [ ]:
# Download MNIST (No change needed if using Google Colab)
myDir = './mnist_data'
train_set = torchvision.datasets.MNIST(myDir, train=True, download=True)
test_set = torchvision.datasets.MNIST(myDir, train=False, download=True)

def to_tensors(dataset):
  # Scale the pixels to [0,1], then normalize with the mean and standard deviation of MNIST
  x = (dataset.data.float() / 255.0 - 0.1307) / 0.3081
  # Add a channel dimension: (N, 28, 28) -> (N, 1, 28, 28)
  return x.unsqueeze(1), dataset.targets.clone()

x_train_all, y_train_all = to_tensors(train_set)
x_test, y_test = to_tensors(test_set)

# Choose a random subset of the training data
n_train = 10000
subset = torch.randperm(len(x_train_all), generator=torch.Generator().manual_seed(0))[:n_train]
x_train = x_train_all[subset].to(device)
y_train = y_train_all[subset].to(device)
x_test = x_test.to(device)
y_test = y_test.to(device)
print("Training images:", tuple(x_train.shape))
print("Test images:", tuple(x_test.shape))

In [ ]:
# Let's draw some of the training data
fig = plt.figure()
for i in range(6):
  plt.subplot(2,3,i+1)
  plt.imshow(x_train[i][0].cpu(), cmap='gray', interpolation='none')
  plt.title("Ground Truth: {}".format(y_train[i].item()))
  plt.xticks([])
  plt.yticks([])
plt.tight_layout()
plt.show()

## Training

The functions below train a network and record how it does. You don't need to change them, but read through `train_model()` so you know what it does:

* It sets the random seed, so every network starts from the same random state and sees the training data in the same order.
* It trains for 5 epochs using SGD (the same settings as notebook 10.5), recording the average training loss every 20 batches.
* It measures the accuracy on the test set at the start and after every epoch.

`plot_results()` draws the training loss and test accuracy of every network you've trained so far, so you can compare them.

In [ ]:
# Training settings -- the same for every network
batch_size = 64
learning_rate = 0.01
momentum = 0.5
n_epochs = 5
log_every = 20   # record the training loss every 20 batches

# He initialization of weights (from notebook 10.5)
def weights_init(layer_in):
  if isinstance(layer_in, nn.Linear):
    nn.init.kaiming_uniform_(layer_in.weight)
    layer_in.bias.data.fill_(0.0)

def count_params(model):
  return sum(p.numel() for p in model.parameters())

# Percent of the test set classified correctly
def evaluate(model):
  model.eval()
  correct = 0
  with torch.no_grad():
    for start in range(0, len(x_test), 1000):
      output = model(x_test[start:start+1000])
      correct += (output.argmax(dim=1) == y_test[start:start+1000]).sum().item()
  return 100.0 * correct / len(x_test)

def train_model(model_class, name):
  # Same random seed for every network
  torch.manual_seed(0)
  model = model_class().to(device)
  model.apply(weights_init)
  print("%s: %d parameters"%(name, count_params(model)))
  optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=momentum)

  history = {'model': model, 'loss_epoch': [], 'train_loss': [], 'test_epoch': [0], 'test_acc': [evaluate(model)]}
  n_batches = int(np.ceil(n_train / batch_size))
  start_time = time.time()
  for epoch in range(n_epochs):
    model.train()
    # Shuffle the training data
    order = torch.randperm(n_train, device=device)
    running_loss = 0.0
    for batch in range(n_batches):
      idx = order[batch * batch_size:(batch + 1) * batch_size]
      optimizer.zero_grad()
      output = model(x_train[idx])
      loss = F.nll_loss(output, y_train[idx])
      loss.backward()
      optimizer.step()
      # Record the average training loss over the last log_every batches
      running_loss += loss.item()
      if (batch + 1) % log_every == 0:
        history['loss_epoch'].append(epoch + (batch + 1) / n_batches)
        history['train_loss'].append(running_loss / log_every)
        running_loss = 0.0
    history['test_epoch'].append(epoch + 1)
    history['test_acc'].append(evaluate(model))
    print("  Epoch %d, test accuracy %3.2f%%  (%.0f seconds)"%(epoch + 1, history['test_acc'][-1], time.time() - start_time))
  return history

# Plot the training loss and test accuracy of every network trained so far
def plot_results(results):
  fig, axes = plt.subplots(1, 2, figsize=(13,4.5))
  for name, history in results.items():
    axes[0].plot(history['loss_epoch'], history['train_loss'], label=name)
    axes[1].plot(history['test_epoch'], history['test_acc'], 'o-', label=name)
  axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Training loss'); axes[0].set_yscale('log')
  axes[0].set_title('Training loss')
  axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Test accuracy (%)')
  axes[1].set_title('Test accuracy')
  for ax in axes:
    ax.set_xlim(0, n_epochs)
    ax.legend()
  plt.show()
  for name, history in results.items():
    print("%-16s final test accuracy %3.2f%%"%(name, history['test_acc'][-1]))

# We'll store the results for every network in this dictionary
results = {}

# **Part 1: CNNs**

First, a small CNN with two convolutional layers. This is similar to the network you built in notebook 10.5, except that we use padding so that each convolution keeps the image the same size, and only the max pools shrink it.

In [ ]:
# TODO -- Fill in this class to implement a CNN with two convolutional layers:
# 1. A convolution with kernel size 3, padding 1, 1 input channel and 16 output channels
# 2. A ReLU
# 3. A max pooling operation over a 2x2 area
# 4. A convolution with kernel size 3, padding 1, 16 input channels and 16 output channels
# 5. A ReLU
# 6. A max pooling operation over a 2x2 area
# 7. A flattening operation
# 8. A fully connected layer mapping from (whatever dimensions we are at -- find out using .shape) to 10
# 9. A log softmax function (use dim=1)
# Define the layers that have parameters (the convolutions and the fully connected layer) in __init__,
# and use them in forward. ReLU, max pooling, flattening, and log softmax can just be called in forward
# (F.relu, F.max_pool2d, x.flatten(1), F.log_softmax).
class TwoConvNet(nn.Module):
  def __init__(self):
    super().__init__()
    # TODO -- define the layers


  def forward(self, x):
    # TODO -- run the input x through the network and return the result

    return x

In [ ]:
# Check the shape of the output before training: it should be (batch size, 10)
print(TwoConvNet()(x_train[0:5].cpu()).shape)

In [ ]:
results['2 conv layers'] = train_model(TwoConvNet, '2 conv layers')
plot_results(results)

Now a deeper network with six convolutional layers. It has more layers, so it can represent more complicated functions.

**Before you run it:** do you expect the 6-layer network to train faster or slower than the 2-layer network over 5 epochs? Will it end up more or less accurate?

In [ ]:
# TODO -- Fill in this class to implement a CNN with six convolutional layers.
# Every convolution has kernel size 3 and padding 1, and is followed by a ReLU.
# 1. conv1: 1 input channel, 16 output channels
# 2. conv2: 16 input channels, 16 output channels, then a 2x2 max pool
# 3. conv3: 16 -> 16
# 4. conv4: 16 -> 16, then a 2x2 max pool
# 5. conv5: 16 -> 16
# 6. conv6: 16 -> 16
# 7. Flatten, a fully connected layer to 10 outputs, and a log softmax (dim=1)
# Please name the layers conv1 ... conv6 and fc -- you'll reuse this structure below.
# Why only two max pools? Each 2x2 max pool halves the size of the image, so pooling after every
# convolution would shrink the 28x28 image to nothing (28 -> 14 -> 7 -> 3 -> 1 -> 0). Pooling twice ends
# at 7x7, the same as TwoConvNet, so both networks can use the same size of fully connected layer.
class SixConvNet(nn.Module):
  def __init__(self):
    super().__init__()
    # TODO -- define the layers


  def forward(self, x):
    # TODO -- run the input x through the network and return the result

    return x

In [ ]:
results['6 conv layers'] = train_model(SixConvNet, '6 conv layers')
plot_results(results)

**Questions:** How do the two training curves compare, especially near the start of training? Which network is more accurate after 5 epochs? Why might a deeper network, which can represent more complicated functions, train more slowly?

# **Part 2: Residual connections**

Now we'll add residual connections to the 6-layer network. In a residual block, the input to the block is added back to its output, so each block only needs to learn a change to its input:

\begin{equation}
\mathbf{h} \leftarrow \mathbf{h} + \text{Conv}\bigl[\text{ReLU}[\mathbf{h}]\bigr]
\end{equation}

This is the same idea as notebook 11.2, but with convolutional layers instead of linear layers. The convolutions use padding, so each block's output is the same size as its input and the two can be added together.

In [ ]:
# The layers are the same as SixConvNet.
# TODO -- Write the forward function so that conv2 ... conv6 are each inside a residual block.
# Use the same order of operations as figure 11.5b in the book (and notebook 11.2):
#   h = h + conv(ReLU(h))
# For example, the first residual block is:  h = h + self.conv2(F.relu(h))
# conv1 is NOT inside a residual block, because it changes the number of channels from 1 to 16,
# so its input and output can't be added together.
# Keep the 2x2 max pools in the same places as SixConvNet (after the conv2 block and after the conv4 block).
# After the last residual block, apply a ReLU, then flatten, the fully connected layer, and a log softmax (dim=1).
class ResNet6(nn.Module):

  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
    self.conv2 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
    self.conv3 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
    self.conv4 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
    self.conv5 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
    self.conv6 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
    self.fc = nn.Linear(16 * 7 * 7, 10)

  def forward(self, x):
    h = self.conv1(x)
    # TODO -- add the residual blocks, max pools, and output layers

    return h

In [ ]:
results['ResNet'] = train_model(ResNet6, 'ResNet')
plot_results(results)

**Questions:** Count the parameters: how many parameters did the residual connections add? How does the residual network's training curve compare to the plain 6-layer network's? Why does adding the input of each block back to its output make the network easier to train?

# **Part 3: Batch normalization**

Finally, we'll add batch normalization. Batch norm normalizes each channel so that it has mean zero and variance one across the batch, then applies a learned scale and shift. In notebook 11.3, you saw that adding residual blocks makes the variance of the activations grow with every block, and that batch normalization keeps it under control.

We won't write batch norm ourselves: PyTorch's `nn.BatchNorm2d` does it for us.

Note that batch norm behaves differently during training (it uses the statistics of the current batch) and testing (it uses running averages saved during training). That's why `train_model()` calls `model.train()` before training and `evaluate()` calls `model.eval()` before testing.

In [ ]:
# TODO -- Add batch normalization to your residual network.
# Following figure 11.6c in the book, apply batch norm at the START of each residual block,
# before the ReLU and the convolution:
#   h = h + conv(ReLU(BatchNorm(h)))
# Use PyTorch's nn.BatchNorm2d. It takes one argument: the number of channels (16 here).
# Each residual block needs its own batch norm layer, because each one learns its own scale and shift.
# Name them bn2 ... bn6 to match the convolutions.
class ResNet6BN(nn.Module):

  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
    self.conv2 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
    self.conv3 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
    self.conv4 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
    self.conv5 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
    self.conv6 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
    self.fc = nn.Linear(16 * 7 * 7, 10)
    # TODO -- define the batch norm layers bn2 ... bn6


  def forward(self, x):
    # TODO -- copy your ResNet6 forward function here and add the batch norm layers

    return x

In [ ]:
results['ResNet + BN'] = train_model(ResNet6BN, 'ResNet + BN')
plot_results(results)

**Questions:** How many parameters does batch norm add, and why? How does the training curve compare to the residual network without batch norm? Which of the four networks would you choose, and why?

**Optional:** Try changing `learning_rate` in the training cell to 0.05 and re-running all four networks. Which networks benefit from the larger learning rate, and which become unstable?

# **Interpret your models**

We know the theory of why residual connections and batch norm should help training, and it appears to have improved the model we can train in five epochs empirically.  Let's try to interpret the differences in our models.  In this cell, fill in the function so that it displays images that the models disagree on - where at least two of the models are wrong.

In [ ]:
def pick_disagreement_image(preds):
  # TODO -- Return the index of a test image where at least two of the models are wrong.
  # preds is a dictionary: preds[name] holds the predicted class of every test image for that model,
  # so (preds[name] != y_test) is True for every test image that model got wrong.
  # If more than one image qualifies (it will), pick one of them at random.
  # Replace this line, which just picks a random test image:
  return np.random.randint(len(x_test))

# ---------- Support code: you don't need to change anything below this line ----------

# Get the predicted class of every test image for every model trained so far
def get_predictions(results):
  preds = {}
  for name, history in results.items():
    model = history['model']
    model.eval()
    with torch.no_grad():
      preds[name] = torch.cat([model(x_test[start:start+1000]).argmax(dim=1) for start in range(0, len(x_test), 1000)])
  return preds

# Show test images, with each model's prediction underneath
def show_images(indices, preds, title=None):
  fig, axes = plt.subplots(1, len(indices), figsize=(3 * len(indices), 4.5))
  axes = np.atleast_1d(axes)
  for ax, index in zip(axes, indices):
    true_label = y_test[index].item()
    ax.imshow(x_test[index][0].cpu(), cmap='gray', interpolation='none')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title("Test image %d\nTrue label: %d"%(index, true_label))
    lines = []
    for name in preds:
      guess = preds[name][index].item()
      lines.append("%s: %d%s"%(name, guess, "" if guess == true_label else " (wrong)"))
    ax.set_xlabel("\n".join(lines), fontsize=8)
  if title is not None:
    fig.suptitle(title)
  plt.tight_layout()
  plt.show()

preds = get_predictions(results)
for name in preds:
  print("%-16s wrong on %d of %d test images"%(name, (preds[name] != y_test).sum().item(), len(y_test)))

# A random test image -- most of the time, every model gets it right
show_images([np.random.randint(len(x_test))], preds, "A random test image")

# Images picked by your function
show_images([pick_disagreement_image(preds) for i in range(5)], preds, "Images the models disagree on")

Do you notice any trends?  Do you have any explanation for the type of images that each network gets wrong or right?  How convinced are you?  Feel free to discuss with your neighbor.

# **Challenge - if you have extra time**

Make a much deeper network, and compare the training curves again.  Alternatively, try to improve your solutions from PA3 using batch norm and residual connections.  Do they help?

In [ ]:
# Your code here